### Structured Output

Structured Output is a technique that forces an LLM to return responses in a predefined format (such as JSON, Pydantic models, dictionaries, or typed objects) instead of free-form text.

### Why Do We Need It?
Consistent responses
Easy parsing by applications
Reduces hallucinated formats
Useful for APIs, Agents, and Automation workflows
Types of Structured Output
### 1. TypedDict

Returns output as a Python dictionary with predefined fields.
class Person(TypedDict):
    name: str
    age: int
### 2. Pydantic Model

Returns validated and type-safe objects.
class Person(BaseModel):
    name: str
    age: int
### 3. JSON Schema

Defines the expected JSON structure for the LLM output.
One-line Interview Answer - Structured Output ensures that LLM responses follow a predefined schema, making them reliable and easy for applications to process.

### TypedDict


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import List,Optional,Annotated,TypedDict
from dotenv import load_dotenv

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)
class Movie(TypedDict):
    title: Annotated[str, "Name of the movie"]
    release_year: Annotated[int, "Year the movie was released"]
    genres: Annotated[List[str], "List of genres the movie belongs to"]
    rating: Annotated[float, "Average rating of the movie on a scale of 1 to 10"]
    box_office: Annotated[Optional[float], "Worldwide box office in millions USD, if known"]

structured_llm = llm.with_structured_output(Movie)
result = structured_llm.invoke("Provide details about the movie 'Interstellar'.")
print(result)
print(type(result))    

{'title': 'Interstellar', 'release_year': 2014, 'genres': ['Sci-Fi', 'Drama', 'Adventure'], 'rating': 8.6, 'box_office': 701.7}
<class 'dict'>


### Pydantic


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import List, Optional
import dotenv


dotenv.load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)

class Movie(BaseModel):
  title: str = Field(..., description="Name of the movie")
  release_year: int = Field(..., description= "Year the movie was released")
  genres: List[str] = Field(..., description= "List of genres the movie belongs to")
  rating: float = Field(..., description= "Average rating of the movie on a scale of 1 to 10")
  box_office: Optional[float] = Field(..., description= "Worldwide box office in millions USD, if known")


#structured_llm = llm.with_structured_output(Movie)


result = llm.with_structured_output(Movie).invoke("Give me details about the movie Interstellar.")

print(result)
print(type(result))
print(result.model_dump()) # converts the Pydantic model instance to a dictionary
print(result.model_dump_json()) # converts the Pydantic model instance to a JSON string, which can be useful for storage or transmission.

movie = Movie(genres=["Sci-Fi", "Action", "Adventure"], 
              title="Inception", 
              release_year=2010, 
              rating=8.8,
              box_office=None)
print(movie)


title='Interstellar' release_year=2014 genres=['Science Fiction', 'Drama', 'Adventure'] rating=8.6 box_office=701.7
<class '__main__.Movie'>
{'title': 'Interstellar', 'release_year': 2014, 'genres': ['Science Fiction', 'Drama', 'Adventure'], 'rating': 8.6, 'box_office': 701.7}
{"title":"Interstellar","release_year":2014,"genres":["Science Fiction","Drama","Adventure"],"rating":8.6,"box_office":701.7}
title='Inception' release_year=2010 genres=['Sci-Fi', 'Action', 'Adventure'] rating=8.8 box_office=None


 ### JSON_Schema

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv

dotenv.load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Correct JSON Schema for LangChain / Gemini function calling
movie_json_schema = {
    "name": "movie_schema",
    "description": "Schema for extracting movie information",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "Name of the movie"
            },
            "release_year": {
                "type": "integer",
                "description": "Year the movie was released"
            },
            "genres": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of genres the movie belongs to"
            },
            "rating": {
                "type": "number",
                "description": "Average rating of the movie on a scale of 1 to 10"
            },
            "box_office": {
                "type": ["number", "null"],
                "description": "Worldwide box office in millions USD, if known"
            }
        },
        "required": ["title", "release_year", "genres", "rating", "box_office"]
    }
}

structured_llm = llm.with_structured_output(movie_json_schema)
result = structured_llm.invoke("Give me details about the movie Inception.")
print(result)

{'title': 'Inception', 'director': 'Christopher Nolan', 'cast': ['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Cillian Murphy', 'Marion Cotillard', 'Michael Caine'], 'release_year': 2010, 'genre': ['Sci-Fi', 'Action', 'Thriller'], 'plot_summary': 'A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into the mind of a C.E.O.', 'imdb_rating': 8.8}


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI


class ProductReview(BaseModel):
    product_name: str = Field(..., description="Name of the product being reviewed")
    reviewer_name: str = Field(..., description="Name of the reviewer")
    rating: float = Field(..., description="Product rating given by the reviewer on a scale of 1 to 5")
    pros: List[str] = Field(..., description="List of positive aspects of the product")
    cons: List[str] = Field(..., description="List of negative aspects of the product")
    review_text: str = Field(..., description="Detailed written review of the product")
    would_recommend: bool = Field(..., description="Whether the reviewer recommends the product")
    purchase_date: Optional[str] = Field(None, description="Date when the product was purchased (YYYY-MM-DD)")


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)
structured_llm = llm.with_structured_output(ProductReview)

result = structured_llm.invoke("""I recently purchased the Bose QuietComfort 45 Noise Cancelling Headphones about three months ago, and I have been using them almost daily since then. As someone who works from home and also travels frequently, I needed headphones that could block out distractions, deliver excellent audio quality, and remain comfortable for long listening sessions. The QC45s absolutely deliver in most of these aspects.

First, the noise cancellation is outstanding. On a recent flight from New York to London, the cabin noise was reduced to a soft whisper, allowing me to enjoy my music and podcasts without cranking up the volume. Even at home, they help me focus by muting the hum of my air conditioner and the occasional street noise.

The sound quality is top-notch. The bass is deep but not overpowering, the mids are clear, and the highs are crisp. Whether I’m listening to jazz, classical music, or electronic tracks, everything sounds balanced and immersive. Bluetooth pairing is seamless, and the connection has been rock solid even when I move around my apartment.

Comfort-wise, I can wear them for 4–5 hours straight without any ear fatigue. The ear cups are soft and well-cushioned, and the clamping force is just right. The build quality feels premium, with smooth adjustments and a sturdy frame.

Battery life is exceptional — I’ve easily gotten over 22 hours of use on a single charge, and the quick-charge feature is a lifesaver when I forget to plug them in overnight.

However, they are not perfect. The touch controls can be a bit finicky, especially in cold weather when I’m wearing gloves. Additionally, the case feels slightly bulky if you’re trying to travel light.

Overall, these headphones have exceeded my expectations. For anyone looking for premium noise-cancelling headphones for travel, work, or daily use, I would highly recommend them without hesitation.
""")
print(result)

product_name='Bose QuietComfort 45 Noise Cancelling Headphones' reviewer_name='Reviewer' rating=5.0 pros=['Outstanding noise cancellation', 'Top-notch sound quality (balanced, immersive audio)', 'Comfortable for long listening sessions (4-5 hours)', 'Premium build quality', 'Exceptional battery life (over 22 hours)', 'Quick-charge feature', 'Seamless and rock-solid Bluetooth pairing'] cons=['Finicky touch controls, especially with gloves', 'Bulky carrying case for travel'] review_text='I recently purchased the Bose QuietComfort 45 Noise Cancelling Headphones about three months ago, and I have been using them almost daily since then. As someone who works from home and also travels frequently, I needed headphones that could block out distractions, deliver excellent audio quality, and remain comfortable for long listening sessions. The QC45s absolutely deliver in most of these aspects.First, the noise cancellation is outstanding. On a recent flight from New York to London, the cabin noise

: 